In [1]:
from typing import List, Optional, Dict, Any
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
import io
import math
import warnings
import os

from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from scipy.stats import genpareto  # used by nothing here; kept for completeness
from sklearn.mixture import GaussianMixture
from sklearn.feature_selection import VarianceThreshold
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import VarianceThreshold
from scipy import stats
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler

try:
    from kneed import KneeLocator
    _HAS_KNEED = True
except Exception:
    _HAS_KNEED = False

In [2]:
class BaseScaler:
    def fit(self, X: np.ndarray):
        raise NotImplementedError
    def transform(self, X: np.ndarray) -> np.ndarray:
        raise NotImplementedError
    def fit_transform(self, X: np.ndarray) -> np.ndarray:
        self.fit(X)
        return self.transform(X)

class SimpleMinMaxScaler(BaseScaler):
    def __init__(self, feature_range=(0.0, 1.0)):
        self._min = None
        self._max = None
        self._range = feature_range
        self._denom = None

    def fit(self, X):
        X = np.asarray(X, dtype=float)
        self._min = np.nanmin(X, axis=0)
        self._max = np.nanmax(X, axis=0)
        self._denom = np.where(self._max - self._min == 0, 1.0, (self._max - self._min))

    def transform(self, X):
        X = np.asarray(X, dtype=float)
        scale = (self._range[1] - self._range[0]) / self._denom
        return self._range[0] + (X - self._min) * scale

class SimpleZScoreScaler(BaseScaler):
    def __init__(self):
        self._mean = None
        self._std = None

    def fit(self, X):
        X = np.asarray(X, dtype=float)
        self._mean = np.nanmean(X, axis=0)
        self._std  = np.nanstd(X, axis=0)
        self._std = np.where(self._std == 0, 1.0, self._std)

    def transform(self, X):
        X = np.asarray(X, dtype=float)
        return (X - self._mean) / self._std

class SimpleRobustScaler(BaseScaler):
    def __init__(self):
        self._median = None
        self._iqr = None

    def fit(self, X):
        X = np.asarray(X, dtype=float)
        # Compute per-column 75th and 25th percentiles
        q75 = np.nanpercentile(X, 75, axis=0)
        q25 = np.nanpercentile(X, 25, axis=0)
        self._median = np.nanmedian(X, axis=0)
        self._iqr = q75 - q25

    def transform(self, X):
        X = np.asarray(X, dtype=float)
        iqr_safe = np.where(self._iqr == 0, 1.0, self._iqr)
        return (X - self._median) / iqr_safe


In [3]:
class Imputer:
    def __init__(self, strategy="mean", columns: Optional[List[str]] = None):
        assert strategy in ("mean", "median", "most_frequent")
        self.strategy = strategy
        self.columns = columns
        self.statistics_: Dict[str, Any] = {}

    def fit(self, df: pd.DataFrame):
        if self.columns is None:
            if self.strategy in ("mean", "median"):
                cols = df.select_dtypes(include=[np.number]).columns.tolist()
            else:
                cols = df.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
        else:
            cols = list(self.columns)

        for col in cols:
            if self.strategy in ("mean", "median"):
                s = pd.to_numeric(df[col], errors='coerce')
                stat = s.mean(skipna=True) if self.strategy == "mean" else s.median(skipna=True)
                if pd.isna(stat):
                    mode = df[col].mode(dropna=True)
                    stat = mode.iloc[0] if not mode.empty else 0.0
                self.statistics_[col] = stat
            else:
                mode = df[col].mode(dropna=True)
                self.statistics_[col] = mode.iloc[0] if not mode.empty else np.nan
        return self

    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        df_copy = df.copy()
        for col, val in self.statistics_.items():
            if col in df_copy.columns:
                df_copy[col].fillna(val, inplace=True)
        return df_copy

    def fit_transform(self, df: pd.DataFrame) -> pd.DataFrame:
        return self.fit(df).transform(df)


In [4]:
# -----------------------------
# Automated Variance Threshold
# Automated variance threshold selection with multiple heuristic/statistical methods.
# Methods available:
# 'arbitrary'         : fixed user-specified threshold (baseline)
# 'percentile'        : keep features above a percentile of variance
# 'elbow'             : elbow/knee detection on sorted variances (kneed fallback)
# 'cross_validation'  : supervised CV to choose threshold (requires y)
# 'information_theory': GMM + AIC to find natural separation
# 'emc_inspired'      : EMC-like separability (unsupervised: CV; supervised: F-ratio)
# -----------------------------
class AutomatedVarianceThreshold:
    def __init__(self, method: str = 'percentile', **kwargs):
        self.method = method
        self.kwargs = kwargs
        self.threshold_: Optional[float] = None
        self.selected_features_: Optional[np.ndarray] = None
        self.variance_scores_: Optional[np.ndarray] = None
        self.selector_info_: Dict[str, Any] = {}

    def _arbitrary_method(self, variances: np.ndarray, threshold: float = 0.01) -> float:
        return float(threshold)

    def _percentile_method(self, variances: np.ndarray, percentile: float = 95.0) -> float:
        return float(np.percentile(variances, percentile))

    def _elbow_method(self, variances: np.ndarray) -> float:
        if not _HAS_KNEED:
            warnings.warn("kneed not available; falling back to percentile(95)")
            return self._percentile_method(variances, percentile=95.0)
        sorted_vars = np.sort(variances)[::-1]
        x_range = list(range(len(sorted_vars)))
        try:
            kneedle = KneeLocator(x_range, sorted_vars, curve="convex", direction="decreasing", interp_method='polynomial')
            if kneedle.knee is not None:
                idx = int(kneedle.knee)
                return float(sorted_vars[idx])
            else:
                return self._percentile_method(variances, percentile=95.0)
        except Exception:
            return self._percentile_method(variances, percentile=95.0)

    def _cross_validation_method(self, X: np.ndarray, y: Optional[np.ndarray] = None, cv_folds: int = 5) -> float:
        if y is None:
            return self._percentile_method(np.var(X, axis=0))
        variances = np.var(X, axis=0)
        thresholds = np.percentile(variances, np.arange(50, 99, 5))
        best_threshold = thresholds[0]
        best_score = -np.inf
        clf = LogisticRegression(random_state=42, max_iter=1000)
        for thr in thresholds:
            sel = variances >= thr
            if sel.sum() < 2:
                continue
            Xs = X[:, sel]
            try:
                scores = cross_val_score(clf, Xs, y, cv=cv_folds, scoring='accuracy')
                mean_score = float(np.mean(scores))
                if mean_score > best_score:
                    best_score = mean_score
                    best_threshold = thr
            except Exception:
                continue
        return float(best_threshold)

    def _information_theory_method(self, variances: np.ndarray) -> float:
        v = np.asarray(variances).reshape(-1, 1)
        best_aic = np.inf
        best_thr = self._percentile_method(variances, percentile=95.0)
        max_components = min(6, max(2, len(variances)//10))
        for n in range(2, max_components+1):
            try:
                gmm = GaussianMixture(n_components=n, random_state=self.kwargs.get('random_state', 42))
                gmm.fit(v)
                aic = gmm.aic(v)
                if aic < best_aic:
                    best_aic = aic
                    means = np.sort(gmm.means_.flatten())
                    if len(means) >= 2:
                        best_thr = float((means[-1] + means[-2]) / 2.0)
            except Exception:
                continue
        return float(best_thr)

    def _emc_inspired_method(self, X: np.ndarray, y: Optional[np.ndarray] = None) -> np.ndarray:
        X = np.asarray(X)
        if y is None:
            means = np.mean(X, axis=0)
            stds = np.std(X, axis=0)
            cv = np.where(np.abs(means) > 0, stds / np.abs(means), stds)
            thr = np.percentile(cv, self.kwargs.get('percentile', 95))
            mask = cv >= thr
            self.selector_info_['cv_scores'] = cv
            self.selector_info_['cv_threshold'] = thr
            return mask
        else:
            y = np.asarray(y)
            labels = np.unique(y)
            separability = []
            for j in range(X.shape[1]):
                col = X[:, j]
                overall_mean = np.mean(col)
                between, within = 0.0, 0.0
                for lab in labels:
                    mask = (y == lab)
                    class_vals = col[mask]
                    c_mean = np.mean(class_vals) if len(class_vals) > 0 else 0.0
                    between += len(class_vals) * (c_mean - overall_mean)**2
                    within += np.sum((class_vals - c_mean)**2)
                score = (between / within) if within > 0 else between
                separability.append(score)
            sep = np.array(separability)
            thr = np.percentile(sep, self.kwargs.get('percentile', 95))
            mask = sep >= thr
            self.selector_info_['separability_scores'] = sep
            self.selector_info_['separability_threshold'] = thr
            return mask

    def fit(self, X: np.ndarray, y: Optional[np.ndarray] = None):
        X = np.asarray(X)
        self.variance_scores_ = np.var(X, axis=0)

        if self.method == 'arbitrary':
            # default = 0.01 unless user provides 'threshold' kwarg
            thr = float(self.kwargs.get('threshold', 0.01))
            self.threshold_ = self._arbitrary_method(self.variance_scores_, thr)
            self.selected_features_ = (self.variance_scores_ >= self.threshold_)

        elif self.method == 'percentile':
            percentile = float(self.kwargs.get('percentile', 95.0))
            self.threshold_ = self._percentile_method(self.variance_scores_, percentile)
            self.selected_features_ = (self.variance_scores_ >= self.threshold_)

        elif self.method == 'elbow':
            self.threshold_ = self._elbow_method(self.variance_scores_)
            self.selected_features_ = (self.variance_scores_ >= self.threshold_)

        elif self.method == 'cross_validation':
            cv_folds = int(self.kwargs.get('cv_folds', 5))
            self.threshold_ = self._cross_validation_method(X, y, cv_folds=cv_folds)
            self.selected_features_ = (self.variance_scores_ >= self.threshold_)

        elif self.method == 'information_theory':
            self.threshold_ = self._information_theory_method(self.variance_scores_)
            self.selected_features_ = (self.variance_scores_ >= self.threshold_)

        elif self.method == 'emc_inspired':
            mask = self._emc_inspired_method(X, y)
            self.selected_features_ = np.asarray(mask, dtype=bool)
            if self.selected_features_.any():
                self.threshold_ = float(np.min(self.variance_scores_[self.selected_features_]))
            else:
                self.threshold_ = float(np.min(self.variance_scores_))

        else:
            raise ValueError(f"Unknown method '{self.method}'")

        return self

    def transform(self, X: np.ndarray) -> np.ndarray:
        if self.selected_features_ is None:
            raise ValueError("Selector not fitted yet.")
        return np.asarray(X)[:, self.selected_features_]

    def fit_transform(self, X: np.ndarray, y: Optional[np.ndarray] = None) -> np.ndarray:
        return self.fit(X, y).transform(X)

    def get_selected_features_mask(self) -> np.ndarray:
        return self.selected_features_

    def get_threshold(self) -> float:
        return float(self.threshold_) if self.threshold_ is not None else None

    def get_feature_scores(self) -> np.ndarray:
        return self.variance_scores_

In [5]:
class TimeSeriesDataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame, window_size: int, num_outputs: int, stride: int = 1, target_column: Optional[str] = None):
        self.df = dataframe.reset_index(drop=True)
        self.window_size = int(window_size)
        self.num_outputs = int(num_outputs)
        self.stride = int(stride)
        if target_column is not None:
            if target_column not in self.df.columns:
                raise ValueError("target_column not found in DataFrame")
            self.target_idx = list(self.df.columns).index(target_column)
        else:
            self.target_idx = 0
        self.data = self.df.values.astype(np.float32)
        max_idx = len(self.data) - self.window_size - self.num_outputs + 1
        self.indices = list(range(0, max(0, max_idx), self.stride)) if max_idx > 0 else []

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        if isinstance(idx, torch.Tensor):
            idx = idx.item()
        start = self.indices[idx]
        x = self.data[start:start + self.window_size]
        y_start = start + self.window_size
        y = self.data[y_start:y_start + self.num_outputs, self.target_idx]
        return torch.from_numpy(x), torch.from_numpy(y)

In [6]:
def analyze_variance_threshold_methods(
    X: np.ndarray, 
    y: Optional[np.ndarray] = None, 
    feature_names: Optional[List[str]] = None
) -> Dict[str, Any]:
    methods = ['arbitrary', 'percentile', 'elbow', 'information_theory', 'emc_inspired']
    if y is not None:
        methods.append('cross_validation')

    results = {}
    for method in methods:
        try:
            if method == 'arbitrary':
                # specify your baseline arbitrary threshold here (e.g., 0.01)
                selector = AutomatedVarianceThreshold(method=method, threshold=0.01)
            else:
                selector = AutomatedVarianceThreshold(method=method)

            # pass parameters if desired
            if method == 'percentile':
                selector.kwargs['percentile'] = 95
            if method == 'emc_inspired':
                selector.kwargs['percentile'] = 95

            X_selected = selector.fit_transform(X, y)
            mask = selector.get_selected_features_mask()
            results[method] = {
                'threshold': selector.get_threshold(),
                'n_features_selected': X_selected.shape[1],
                'selected_features_mask': mask,
                'feature_scores': selector.get_feature_scores()
            }
            if feature_names is not None:
                results[method]['selected_feature_names'] = [
                    n for n, m in zip(feature_names, mask) if m
                ]
        except Exception as e:
            results[method] = {'error': str(e)}

    return results

In [7]:
def run_pipeline_on_telco_file(telco_path: str, output_dir: Optional[str] = None):
    df = pd.read_csv(telco_path)
    # ensure TotalCharges numeric
    if 'TotalCharges' in df.columns:
        df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

    # drop identifier columns commonly present
    for idcol in ['customerID', 'CustomerID', 'id', 'Id']:
        if idcol in df.columns:
            df = df.drop(columns=[idcol])

    # identify categorical columns (exclude target 'Churn' if present)
    exclude = ['Churn']
    cat_cols = [c for c in df.select_dtypes(include=['object','category','bool']).columns if c not in exclude]

    # impute numeric columns (median)
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    num_imputer = Imputer(strategy="median", columns=numeric_cols)
    df = num_imputer.fit_transform(df)

    # impute categorical (most frequent)
    if cat_cols:
        cat_imputer = Imputer(strategy="most_frequent", columns=cat_cols)
        df = cat_imputer.fit_transform(df)

    # one-hot encode categorical columns
    df_ohe = pd.get_dummies(df, columns=cat_cols, drop_first=False, dtype=int)

    # safety fill remaining NaNs with numeric medians
    if df_ohe.isna().values.any():
        df_ohe = df_ohe.fillna(df_ohe.median(numeric_only=True))

    # choose numeric columns (exclude Churn if present)
    numeric_cols = [c for c in df_ohe.columns if df_ohe[c].dtype.kind in "fi" and c != 'Churn']

    arb_thresh = 0.01
    selector_raw = VarianceThreshold(threshold=arb_thresh)
    X_raw = df_ohe[numeric_cols].to_numpy()
    X_sel_raw = selector_raw.fit_transform(X_raw)

    kept_count_raw = X_sel_raw.shape[1]
    total_raw = X_raw.shape[1]
    frac_var_kept_raw = (df_ohe[numeric_cols].var(axis=0, ddof=0)
                     .loc[selector_raw.get_support()].sum()
                     / df_ohe[numeric_cols].var(axis=0, ddof=0).sum())

    print(f"[Raw] Kept {kept_count_raw}/{total_raw} features "
          f"({frac_var_kept_raw:.3f} variance kept)")

    scalers = {
        'minmax': SimpleMinMaxScaler(feature_range=(0.0, 1.0)),
        'zscore': SimpleZScoreScaler(),
        'robust': SimpleRobustScaler()
    }

    for name, scaler in scalers.items():
        Xnum = df_ohe[numeric_cols].to_numpy(dtype=float)
        scaler.fit(Xnum)
        Xs = scaler.transform(Xnum)

    
        selector = VarianceThreshold(threshold=arb_thresh)
        X_sel = selector.fit_transform(Xs)
    
        kept_count = X_sel.shape[1]
        total = Xs.shape[1]
        frac_var_kept = (pd.Series(Xs.var(axis=0))
                     .loc[selector.get_support()].sum()
                     / pd.Series(Xs.var(axis=0)).sum())
    
        print(f"[{name}] Kept {kept_count}/{total} features "
              f"({frac_var_kept:.3f} variance kept)")


    # run each scaler and several selector methods
    overall_reports = {}
    for scaler_name, scaler in scalers.items():
        print(f"\n--- Scaler: {scaler_name} ---")
        Xnum = df_ohe[numeric_cols].to_numpy(dtype=float)
        scaler.fit(Xnum)
        Xs = scaler.transform(Xnum)
        scaled_df = pd.DataFrame(Xs, columns=numeric_cols, index=df_ohe.index)
        other_cols = [c for c in df_ohe.columns if c not in numeric_cols]
        combined = pd.concat([scaled_df, df_ohe[other_cols].reset_index(drop=True)], axis=1)

        # Compute numeric variances (only numeric columns)
        numeric_combined = combined.select_dtypes(include=[np.number])
        var_series = numeric_combined.var(axis=0, ddof=0)

        scaler_reports = {}

        arb_thresh = 0.01
        selector = VarianceThreshold(threshold=arb_thresh)
        try:
            X_sel = selector.fit_transform(numeric_combined.to_numpy())
            mask = selector.get_support()
            names_num = numeric_combined.columns.tolist()
            kept = [n for n, m in zip(names_num, mask) if m]
            removed = [n for n, m in zip(names_num, mask) if not m]
            kept_count = len(kept)
            total = len(names_num)
            fraction_var_kept = var_series.loc[kept].sum() / var_series.sum() if var_series.sum() > 0 else np.nan
            scaler_reports['arbitrary'] = {
                'threshold': arb_thresh,
                'kept': kept,
                'removed': removed,
                'kept_count': kept_count,
                'total_numeric': total,
                'fraction_variance_kept': fraction_var_kept
            }
            print(f"Method=arbitrary        -> threshold={arb_thresh:.6g}, kept {kept_count}/{total}")
            pruned_df = combined.loc[:, kept + ([c for c in other_cols if c == 'Churn'])]
            if output_dir:
                outname = f"{output_dir}/telco_pruned_{scaler_name}_arbitrary.csv"
                pruned_df.to_csv(outname, index=False)
        except Exception as e:
            scaler_reports['arbitrary'] = {'error': str(e)}
            print(f"Method=arbitrary        -> ERROR: {e}")

        methods_to_try = ['percentile', 'elbow', 'information_theory', 'emc_inspired']
        if 'Churn' in combined.columns:
            methods_to_try.append('cross_validation')

        for method in methods_to_try:
            try:
                selector = AutomatedVarianceThreshold(method=method)
                y = combined['Churn'].to_numpy() if (method == 'cross_validation' and 'Churn' in combined.columns) else None
                selector.fit(numeric_combined.to_numpy(), y)
                mask = selector.get_selected_features_mask()
                names_num = numeric_combined.columns.tolist()
                kept = [n for n, m in zip(names_num, mask) if m]
                removed = [n for n, m in zip(names_num, mask) if not m]
                kept_count = len(kept)
                total = len(names_num)
                fraction_var_kept = var_series.loc[kept].sum() / var_series.sum() if var_series.sum() > 0 else np.nan
                scaler_reports[method] = {
                    'threshold': selector.get_threshold(),
                    'kept': kept,
                    'removed': removed,
                    'kept_count': kept_count,
                    'total_numeric': total,
                    'fraction_variance_kept': fraction_var_kept
                }
                print(f"Method={method:17} -> threshold={selector.get_threshold():.6g}, kept {kept_count}/{total}")
                pruned_df = combined.loc[:, kept + ([c for c in other_cols if c == 'Churn'])]
                if output_dir:
                    outname = f"{output_dir}/telco_pruned_{scaler_name}_{method}.csv"
                    pruned_df.to_csv(outname, index=False)
            except Exception as e:
                scaler_reports[method] = {'error': str(e)}
                print(f"Method={method:17} -> ERROR: {e}")

        overall_reports[scaler_name] = scaler_reports

    return overall_reports

In [8]:
def main():

    # 2) Telco dataset pipeline
    telco_path = "telco_customer_churn.csv"
    if not os.path.exists(telco_path):
        print("\nNo telco_customer_churn.csv found in working directory. Skipping Telco run.")
    else:
        print("\n Running pipeline on Telco dataset")
        df = pd.read_csv(telco_path)

        # preprocess
        if 'TotalCharges' in df.columns:
            df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
        for idcol in ['customerID','CustomerID','id','Id']:
            if idcol in df.columns:
                df = df.drop(columns=[idcol])

        exclude = ['Churn']
        cat_cols = [c for c in df.select_dtypes(include=['object','category','bool']).columns if c not in exclude]

        num_imputer = Imputer(strategy="median", columns=df.select_dtypes(include=[np.number]).columns.tolist())
        df = num_imputer.fit_transform(df)
        if cat_cols:
            cat_imputer = Imputer(strategy="most_frequent", columns=cat_cols)
            df = cat_imputer.fit_transform(df)

        df_ohe = pd.get_dummies(df, columns=cat_cols, drop_first=False, dtype=int)
        if df_ohe.isna().values.any():
            df_ohe = df_ohe.fillna(df_ohe.median(numeric_only=True))

        # labels
        has_churn = 'Churn' in df_ohe.columns
        y = None
        if has_churn:
            y_raw = df_ohe['Churn']
            if y_raw.dtype.kind in "O":
                y = (y_raw.astype(str).str.lower() == 'yes').astype(int).to_numpy()
            else:
                y = y_raw.to_numpy().astype(int)
            print("Churn label distribution:", np.bincount(y))

        # numeric cols
        numeric_cols = [c for c in df_ohe.columns if df_ohe[c].dtype.kind in "fi" and c != 'Churn']
        print("Number of numeric columns considered:", len(numeric_cols))

        arb_thresh = 0.01
        selector_raw = VarianceThreshold(threshold=arb_thresh)
        X_raw = df_ohe[numeric_cols].to_numpy(dtype=float)
        selector_raw.fit(X_raw)
        mask_raw = selector_raw.get_support()

        kept_cols_raw = [numeric_cols[i] for i, m in enumerate(mask_raw) if m]
        kept_count_raw = len(kept_cols_raw)
        total_raw = len(numeric_cols)
        frac_var_kept_raw = (df_ohe[numeric_cols].var(axis=0, ddof=0)
                             .loc[kept_cols_raw].sum()
                             / df_ohe[numeric_cols].var(axis=0, ddof=0).sum())
        print(f"[Raw] Kept {kept_count_raw}/{total_raw} features "
              f"({frac_var_kept_raw:.3f} variance kept)")

        rows = []

        # scalers
        scalers = {
            'minmax': SimpleMinMaxScaler(feature_range=(0.0,1.0)),
            'zscore': SimpleZScoreScaler(),
            'robust': SimpleRobustScaler()
        }

        # methods (add arbitrary)
        methods = ['arbitrary','percentile','elbow','information_theory','emc_inspired']
        if has_churn:
            methods.append('cross_validation')

        rows = []
        for scaler_name, scaler in scalers.items():
            print("\nRunning scaler:", scaler_name)
            Xnum = df_ohe[numeric_cols].to_numpy(dtype=float)
            scaler.fit(Xnum)
            Xs = scaler.transform(Xnum)
            scaled_numeric_df = pd.DataFrame(Xs, columns=numeric_cols, index=df_ohe.index)

            var_series = scaled_numeric_df.var(axis=0, ddof=0)
            total_variance = var_series.sum()

            for method in methods:
                try:
                    if method == 'arbitrary':
                        selector = VarianceThreshold(threshold=0.01)
                        selector.fit(scaled_numeric_df.to_numpy())
                        mask = selector.get_support()
                        threshold = 0.01
                    else:
                        selector = AutomatedVarianceThreshold(method=method)
                        selector.fit(scaled_numeric_df.to_numpy(), y if method=='cross_validation' else None)
                        mask = selector.get_selected_features_mask()
                        threshold = selector.get_threshold()

                    kept_indices = np.where(mask)[0]
                    kept_cols = [scaled_numeric_df.columns[i] for i in kept_indices]
                    kept_count = len(kept_cols)

                    # Fraction variance kept
                    frac_var_kept = (var_series.loc[kept_cols].sum() / total_variance) if (total_variance > 0 and kept_count>0) else np.nan

                    # Evaluate classifier
                    acc_mean = np.nan
                    auc_mean = np.nan
                    if has_churn and kept_count >= 2:
                        X_pruned = scaled_numeric_df.loc[:, kept_cols].to_numpy()
                        clf = LogisticRegression(max_iter=1000, solver='liblinear')
                        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
                        acc = cross_val_score(clf, X_pruned, y, cv=skf, scoring='accuracy')
                        auc = cross_val_score(clf, X_pruned, y, cv=skf, scoring='roc_auc')
                        acc_mean = float(np.mean(acc))
                        auc_mean = float(np.mean(auc))

                    rows.append({
                        'scaler': scaler_name,
                        'method': method,
                        'threshold': threshold,
                        'kept_count': kept_count,
                        'total_numeric': len(scaled_numeric_df.columns),
                        'fraction_variance_kept': frac_var_kept,
                        'accuracy_cv': acc_mean,
                        'auc_cv': auc_mean
                    })
                    print(f"  method={method:17} kept={kept_count:2d} frac_var={frac_var_kept:0.3f} acc={acc_mean:.3f} auc={auc_mean:.3f}")

                except Exception as e:
                    print(f"  method={method:17} ERROR: {e}")
                    rows.append({
                        'scaler': scaler_name,
                        'method': method,
                        'threshold': None,
                        'kept_count': None,
                        'total_numeric': len(scaled_numeric_df.columns),
                        'fraction_variance_kept': None,
                        'accuracy_cv': None,
                        'auc_cv': None,
                        'error': str(e)
                    })

        # Compact results DataFrame
        df_results = pd.DataFrame(rows)
        expected_cols = ['scaler','method','threshold','kept_count','total_numeric',
                         'fraction_variance_kept','accuracy_cv','auc_cv','error']
        for col in expected_cols:
            if col not in df_results.columns:
                df_results[col] = np.nan
        df_results = df_results[expected_cols].fillna(np.nan)

    # TimeSeries
    ts_path = "electric_production.csv"
    if not os.path.exists(ts_path):
        print("\nNo electric_production.csv found - skipping time series demo.")
    else:
        print("\n Time Series Dataset")
        df_ts = pd.read_csv(ts_path)
        numeric_df = df_ts.select_dtypes(include=[np.number]).dropna(axis=1, how='all')
        ds = TimeSeriesDataset(numeric_df, window_size=12, num_outputs=3, stride=1)
        print(f"TimeSeriesDataset length: {len(ds)}")
        if len(ds) > 0:
            x, y = ds[0]
            print("Sample x shape:", x.shape, "Sample y shape:", y.shape)


if __name__ == "__main__":
    main()


 Running pipeline on Telco dataset
Churn label distribution: [5174 1869]
Number of numeric columns considered: 45
[Raw] Kept 45/45 features (1.000 variance kept)

Running scaler: minmax


C:\Users\Dianne Yumol\AppData\Local\Temp\ipykernel_20096\1262510881.py:34: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_copy[col].fillna(val, inplace=True)


  method=arbitrary         kept=45 frac_var=1.000 acc=0.803 auc=0.844
  method=percentile        kept= 3 frac_var=0.085 acc=0.735 auc=0.697
  method=elbow             kept= 1 frac_var=0.028 acc=nan auc=nan
  method=information_theory kept=18 frac_var=0.498 acc=0.784 auc=0.822
  method=emc_inspired      kept= 3 frac_var=0.035 acc=0.735 auc=0.561
  method=cross_validation  kept=24 frac_var=0.646 acc=0.787 auc=0.826

Running scaler: zscore
  method=arbitrary         kept=45 frac_var=1.000 acc=0.805 auc=0.845
  method=percentile        kept= 3 frac_var=0.067 acc=0.731 auc=0.705
  method=elbow             kept= 3 frac_var=0.067 acc=0.731 auc=0.705
  method=information_theory kept=45 frac_var=1.000 acc=0.805 auc=0.845
  method=emc_inspired      kept= 3 frac_var=0.067 acc=0.735 auc=0.627
  method=cross_validation  kept=29 frac_var=0.644 acc=0.793 auc=0.836

Running scaler: robust
  method=arbitrary         kept=45 frac_var=1.000 acc=0.805 auc=0.845
  method=percentile        kept= 3 frac_var=

Q: Assess the quality of the dataset before normalization and after normalization (1 non-normalized dataset and 3 normalized datasets)

In [10]:
class DatasetQualityAssessment:
    def __init__(self, data, dataset_name="Dataset"):
        self.data = data
        self.dataset_name = dataset_name
        self.numeric_cols = data.select_dtypes(include=[np.number]).columns.tolist()
        
    def basic_info(self):
        # Basic dataset information
        print(f"\n{'='*50}")
        print(f"BASIC INFO: {self.dataset_name}")
        print(f"Shape: {self.data.shape}")
        print(f"Numeric columns: {len(self.numeric_cols)}")
        print(f"Non-numeric columns: {self.data.shape[1] - len(self.numeric_cols)}")
        print(f"Total missing values: {self.data.isnull().sum().sum()}")
        print(f"Memory usage: {self.data.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
        
    def summary_statistics(self):
        # Comprehensive summary statistics
        print(f"\n{'='*50}")
        print(f"SUMMARY STATISTICS: {self.dataset_name}")
        
        if len(self.numeric_cols) == 0:
            print("No numeric columns found.")
            return None
            
        # Basic statistics
        desc = self.data[self.numeric_cols].describe()
        print("\nBasic Statistics:")
        print(desc.round(4))
        
        # Additional statistics
        additional_stats = pd.DataFrame(index=self.numeric_cols)
        additional_stats['missing_count'] = self.data[self.numeric_cols].isnull().sum()
        additional_stats['missing_pct'] = (self.data[self.numeric_cols].isnull().sum() / len(self.data)) * 100
        additional_stats['unique_values'] = self.data[self.numeric_cols].nunique()
        additional_stats['skewness'] = self.data[self.numeric_cols].skew()
        additional_stats['kurtosis'] = self.data[self.numeric_cols].kurtosis()
        
        # Range and scale information
        additional_stats['range'] = self.data[self.numeric_cols].max() - self.data[self.numeric_cols].min()
        additional_stats['coefficient_of_variation'] = (self.data[self.numeric_cols].std() / self.data[self.numeric_cols].mean()).abs()
        
        print("\nAdditional Statistics:")
        print(additional_stats.round(4))
        
        return desc, additional_stats
    
    def outlier_analysis(self):
        # Detect and analyze outliers using multiple methods
        print(f"OUTLIER ANALYSIS: {self.dataset_name}")
        
        outlier_summary = pd.DataFrame(index=self.numeric_cols)
        
        for col in self.numeric_cols:
            data_col = self.data[col].dropna()
            
            # IQR method
            Q1 = data_col.quantile(0.25)
            Q3 = data_col.quantile(0.75)
            IQR = Q3 - Q1
            lower_bound = Q1 - 1.5 * IQR
            upper_bound = Q3 + 1.5 * IQR
            iqr_outliers = ((data_col < lower_bound) | (data_col > upper_bound)).sum()
            
            # Z-score method (|z| > 3)
            z_scores = np.abs(stats.zscore(data_col))
            zscore_outliers = (z_scores > 3).sum()
            
            # Modified Z-score method (using MAD)
            median = np.median(data_col)
            mad = np.median(np.abs(data_col - median))
            modified_z_scores = 0.6745 * (data_col - median) / mad if mad != 0 else np.zeros_like(data_col)
            modified_zscore_outliers = (np.abs(modified_z_scores) > 3.5).sum()
            
            outlier_summary.loc[col, 'iqr_outliers'] = iqr_outliers
            outlier_summary.loc[col, 'iqr_outliers_pct'] = (iqr_outliers / len(data_col)) * 100
            outlier_summary.loc[col, 'zscore_outliers'] = zscore_outliers
            outlier_summary.loc[col, 'zscore_outliers_pct'] = (zscore_outliers / len(data_col)) * 100
            outlier_summary.loc[col, 'modified_zscore_outliers'] = modified_zscore_outliers
            outlier_summary.loc[col, 'modified_zscore_outliers_pct'] = (modified_zscore_outliers / len(data_col)) * 100
        
        print(outlier_summary.round(2))
        return outlier_summary
    
    def distribution_analysis(self):
        # Analyze distributions and normality
        print(f"DISTRIBUTION ANALYSIS: {self.dataset_name}")
        
        dist_analysis = pd.DataFrame(index=self.numeric_cols)
        
        for col in self.numeric_cols:
            data_col = self.data[col].dropna()
            
            if len(data_col) < 3:
                continue
                
            # Shapiro-Wilk test for normality (for small samples)
            if len(data_col) <= 5000:
                shapiro_stat, shapiro_p = stats.shapiro(data_col)
                dist_analysis.loc[col, 'shapiro_stat'] = shapiro_stat
                dist_analysis.loc[col, 'shapiro_pvalue'] = shapiro_p
                dist_analysis.loc[col, 'is_normal_shapiro'] = shapiro_p > 0.05
            
            # Kolmogorov-Smirnov test for normality
            ks_stat, ks_p = stats.kstest(data_col, 'norm', args=(data_col.mean(), data_col.std()))
            dist_analysis.loc[col, 'ks_stat'] = ks_stat
            dist_analysis.loc[col, 'ks_pvalue'] = ks_p
            dist_analysis.loc[col, 'is_normal_ks'] = ks_p > 0.05
            
        print(dist_analysis.round(4))
        return dist_analysis
    
    def scale_analysis(self):
        # Analyze feature scales and ranges
        print(f"SCALE ANALYSIS: {self.dataset_name}")
        
        scale_info = pd.DataFrame(index=self.numeric_cols)
        
        for col in self.numeric_cols:
            data_col = self.data[col].dropna()
            
            scale_info.loc[col, 'min'] = data_col.min()
            scale_info.loc[col, 'max'] = data_col.max()
            scale_info.loc[col, 'range'] = data_col.max() - data_col.min()
            scale_info.loc[col, 'mean'] = data_col.mean()
            scale_info.loc[col, 'std'] = data_col.std()
            scale_info.loc[col, 'magnitude_order'] = np.floor(np.log10(np.abs(data_col.mean()))) if data_col.mean() != 0 else 0
        
        # Calculate scale differences
        ranges = scale_info['range'].dropna()
        if len(ranges) > 0:
            scale_info['range_ratio_to_max'] = ranges / ranges.max()
            scale_info['range_ratio_to_median'] = ranges / ranges.median()
        
        print(scale_info.round(4))
        return scale_info
    
    def correlation_analysis(self):
        # Analyze correlations between features
        print(f"CORRELATION ANALYSIS: {self.dataset_name}")
        
        if len(self.numeric_cols) < 2:
            print("Need at least 2 numeric columns for correlation analysis")
            return None
            
        corr_matrix = self.data[self.numeric_cols].corr()
        
        # Find high correlations
        high_corr_pairs = []
        for i in range(len(corr_matrix.columns)):
            for j in range(i+1, len(corr_matrix.columns)):
                corr_val = corr_matrix.iloc[i, j]
                if abs(corr_val) > 0.7:  # High correlation threshold
                    high_corr_pairs.append({
                        'feature1': corr_matrix.columns[i],
                        'feature2': corr_matrix.columns[j],
                        'correlation': corr_val
                    })
        
        print(f"High correlations (|r| > 0.7): {len(high_corr_pairs)} pairs found")
        if high_corr_pairs:
            high_corr_df = pd.DataFrame(high_corr_pairs)
            high_corr_df = high_corr_df.sort_values('correlation', key=abs, ascending=False)
            print(high_corr_df.head(10))
        
        # Correlation statistics
        corr_values = corr_matrix.values[np.triu_indices_from(corr_matrix.values, k=1)]
        print(f"\nCorrelation Statistics:")
        print(f"Mean absolute correlation: {np.mean(np.abs(corr_values)):.4f}")
        print(f"Max absolute correlation: {np.max(np.abs(corr_values)):.4f}")
        print(f"Std of correlations: {np.std(corr_values):.4f}")
        
        return corr_matrix, high_corr_pairs
    
    def data_quality_score(self):
        print(f"DATA QUALITY SCORE: {self.dataset_name}")
        
        scores = {}
        
        # Completeness score (based on missing values)
        total_cells = self.data.shape[0] * self.data.shape[1]
        missing_cells = self.data.isnull().sum().sum()
        completeness = (total_cells - missing_cells) / total_cells
        scores['completeness'] = completeness
        
        # Uniqueness score (based on duplicate rows)
        duplicates = self.data.duplicated().sum()
        uniqueness = (len(self.data) - duplicates) / len(self.data)
        scores['uniqueness'] = uniqueness
        
        # Scale consistency score (for numeric data)
        if self.numeric_cols:
            ranges = []
            for col in self.numeric_cols:
                col_range = self.data[col].max() - self.data[col].min()
                if not np.isnan(col_range) and col_range > 0:
                    ranges.append(col_range)
            
            if ranges:
                range_cv = np.std(ranges) / np.mean(ranges) if np.mean(ranges) > 0 else 0
                scale_consistency = max(0, 1 - range_cv)  # Lower CV = better consistency
                scores['scale_consistency'] = scale_consistency
        
        # Overall quality score
        overall_score = np.mean(list(scores.values()))
        scores['overall_quality'] = overall_score
        
        print("Quality Scores (0-1, higher is better):")
        for metric, score in scores.items():
            print(f"{metric.replace('_', ' ').title()}: {score:.4f}")
        
        return scores

def compare_datasets(datasets_dict):
    print(f"\n{'='*60}")
    print("DATASET COMPARISON SUMMARY")
    print(f"{'='*60}")
    
    comparison_df = pd.DataFrame()
    
    for name, df in datasets_dict.items():
        assessor = DatasetQualityAssessment(df, name)
        scores = assessor.data_quality_score()
        
        # Add basic info
        comparison_df.loc[name, 'rows'] = df.shape[0]
        comparison_df.loc[name, 'columns'] = df.shape[1]
        comparison_df.loc[name, 'numeric_columns'] = len(df.select_dtypes(include=[np.number]).columns)
        comparison_df.loc[name, 'missing_values'] = df.isnull().sum().sum()
        comparison_df.loc[name, 'missing_pct'] = (df.isnull().sum().sum() / (df.shape[0] * df.shape[1])) * 100
        
        # Add quality scores
        for metric, score in scores.items():
            comparison_df.loc[name, f'quality_{metric}'] = score
        
        # Add scale information for numeric columns
        numeric_cols = df.select_dtypes(include=[np.number]).columns
        if len(numeric_cols) > 0:
            ranges = []
            for col in numeric_cols:
                col_range = df[col].max() - df[col].min()
                if not np.isnan(col_range):
                    ranges.append(col_range)
            
            if ranges:
                comparison_df.loc[name, 'mean_range'] = np.mean(ranges)
                comparison_df.loc[name, 'std_range'] = np.std(ranges)
                comparison_df.loc[name, 'max_range'] = np.max(ranges)
                comparison_df.loc[name, 'min_range'] = np.min(ranges)
    
    print("\nDataset Comparison:")
    print(comparison_df.round(4))
    
    return comparison_df

def analyze_single_dataset(df, dataset_name="Dataset"):
    assessor = DatasetQualityAssessment(df, dataset_name)
    
    assessor.basic_info()
    desc, additional = assessor.summary_statistics()
    outliers = assessor.outlier_analysis()
    distributions = assessor.distribution_analysis()
    scales = assessor.scale_analysis()
    correlations = assessor.correlation_analysis()
    quality_scores = assessor.data_quality_score()
    
    return {
        'basic_stats': desc,
        'additional_stats': additional,
        'outliers': outliers,
        'distributions': distributions,
        'scales': scales,
        'correlations': correlations,
        'quality_scores': quality_scores
    }

def create_normalization_comparison(original_df, target_columns=None):
    if target_columns is None:
        target_columns = original_df.select_dtypes(include=[np.number]).columns.tolist()
    
    # Create normalized versions
    datasets = {'Original': original_df}
    
    # MinMax Scaler (0-1)
    minmax_scaler = MinMaxScaler()
    df_minmax = original_df.copy()
    df_minmax[target_columns] = minmax_scaler.fit_transform(original_df[target_columns])
    datasets['MinMax_Normalized'] = df_minmax
    
    # Standard Scaler (Z-score)
    standard_scaler = StandardScaler()
    df_standard = original_df.copy()
    df_standard[target_columns] = standard_scaler.fit_transform(original_df[target_columns])
    datasets['ZScore_Normalized'] = df_standard
    
    # Robust Scaler
    robust_scaler = RobustScaler()
    df_robust = original_df.copy()
    df_robust[target_columns] = robust_scaler.fit_transform(original_df[target_columns])
    datasets['Robust_Normalized'] = df_robust
    
    # Analyze each dataset
    all_results = {}
    for name, df in datasets.items():
        print(f"ANALYZING: {name}")
        all_results[name] = analyze_single_dataset(df, name)
    
    # Compare all datasets
    comparison = compare_datasets(datasets)
    
    return datasets, all_results, comparison

df = pd.read_csv('telco_customer_churn.csv')
results = analyze_single_dataset(df, 'telco_customer_churn.csv')
datasets, all_results, comparison = create_normalization_comparison(df)


BASIC INFO: telco_customer_churn.csv
Shape: (7043, 21)
Numeric columns: 3
Non-numeric columns: 18
Total missing values: 0
Memory usage: 6.82 MB

SUMMARY STATISTICS: telco_customer_churn.csv

Basic Statistics:
       SeniorCitizen     tenure  MonthlyCharges
count      7043.0000  7043.0000       7043.0000
mean          0.1621    32.3711         64.7617
std           0.3686    24.5595         30.0900
min           0.0000     0.0000         18.2500
25%           0.0000     9.0000         35.5000
50%           0.0000    29.0000         70.3500
75%           0.0000    55.0000         89.8500
max           1.0000    72.0000        118.7500

Additional Statistics:
                missing_count  missing_pct  unique_values  skewness  kurtosis  \
SeniorCitizen               0          0.0              2    1.8336    1.3626   
tenure                      0          0.0             73    0.2395   -1.3874   
MonthlyCharges              0          0.0           1585   -0.2205   -1.2573   

         

MinMax normalization emerged as the best overall choice for our dataset, demonstrating perfect scale consistency by transforming all features to a uniform 0-1 range while preserving the underlying data relationships and maintaining relative distances between data points. This method handles outliers effectively, as evidenced by our max_range=1.0 result showing clean normalization, and it maintains excellent interpretability where 0 represents the minimum value and 1 represents the maximum value for each feature. 

In contrast, Z-score normalization, while good for algorithms that assume normal distributions by centering the data, shows some concerning characteristics including wider range variation with max_range=3.34 that suggests some features contain extreme outliers, different scales per feature since each has its own standard deviation, and reduced interpretability due to negative values and varying scales across features. 

Robust normalization offers moderate performance with better scale consistency than Z-score (max_range=1.85) but not as clean as MinMax, though it provides valuable outlier resistance by using median and interquartile range instead of mean and standard deviation, making it particularly suitable for skewed data like ours, despite still showing some scale differences with std_range=0.35 indicating residual variability.